<a href="https://colab.research.google.com/github/mdkamrulhasan/data_mining_kdd/blob/main/notebooks/Basic_Statistics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CIS 635 — Knowledge Discovery & Data Mining
## Basic Statistics for Data Science

Welcome! This notebook accompanies the *Basic Statistics* lecture and is meant to be worked through top to bottom in **Google Colab**.

Statistics is the foundation of data mining: before we can discover patterns in data, cluster it, or build predictive models, we need a shared vocabulary for describing data and summarizing it honestly. This notebook builds that vocabulary from the ground up, using the same running example from lecture (a small package of "weights") plus a few simulated datasets so you can see the ideas in action.

**What you'll cover in this notebook:**

1. Why statistics matters — a motivating scenario
2. What "statistics" actually means
3. Types of data: numerical vs. categorical
4. Data sets, samples, and variables
5. Population vs. sample
6. Statistic vs. parameter, and the idea of a census
7. Bias in data collection
8. Measures of central tendency: mean, median, mode
9. Variance and standard deviation
10. Percentiles
11. **Exercises** — five short problems to check your understanding

**How to use this notebook:** Read each markdown (text) cell, then run the code cell(s) that follow it (Shift+Enter). Every code cell has a comment above it explaining what it demonstrates and why. Feel free to change the numbers and re-run cells to build intuition — that's the best way to learn statistics.

*Reference: Chapter 4, "Statistics for Dummies" (Rumsey), as used in the CIS 635 lecture slides.*


## Setup

We'll use a small, standard set of libraries throughout this notebook:

- **numpy** for numerical arrays and simulating data
- **pandas** for organizing data into tables (DataFrames)
- **matplotlib** for plotting
- **scipy.stats** for a couple of statistical helper functions (e.g., mode)

Run the cell below once at the start of your session.


In [ ]:
# Import the core libraries we'll use throughout the notebook, and fix a
# random seed so that every "random" simulation below is reproducible —
# you should get the exact same numbers every time you re-run this notebook.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
plt.rcParams['figure.figsize'] = (7, 4)

print("Libraries loaded. NumPy version:", np.__version__)


## 1. Why Statistics? A Motivating Scenario

Imagine you're a data scientist working for a **drug-discovery company**:

- Your company is planning (both long- and short-term) to develop a new drug.
- A **competitor** already has a very popular drug that has been on the market for **10 years**, with a reported **efficacy rate of ~90%**.

**Discussion questions (no code needed — just think it through, or jot notes in a markdown cell):**

- What are your first thoughts on this problem?
- How would you even begin to approach it?
- What would you want to know about *how* that 90% number was measured before trusting it?

Keep this scenario in mind — nearly every concept below (samples, bias, central tendency, variance) is a tool for answering exactly these kinds of questions responsibly.


### What Is "Statistics," Really?

> In today's world, the buzzword is *data*, as in: "Do you have any data to support your claim?" "What data do you have on this?" "The data supported the original hypothesis that ...," "Statistical data show that ...," and "The data bear this out ...." But the field of statistics is not just about data.

**Statistics** is the entire process involved in **gathering evidence to answer questions about the world**, in cases where that evidence happens to be data. It's not just numbers — it's the full pipeline: asking a good question, collecting data carefully, summarizing it fairly, and drawing a defensible conclusion.


## 2. Types of Data

At the lowest level, almost all data falls into one of two groups:

- **Numerical (Quantitative) data** — a measurement or a count.
  - *Measurements:* a person's height, weight, or blood pressure.
  - *Counts:* the number of shares someone owns, the number of teeth a dog has, the number of pages in a book.
  - Numerical data is further split into:
    - **Discrete**: countable values you could list out (e.g., number of siblings: 0, 1, 2, 3, ...).
    - **Continuous**: measurements whose possible values can't be counted/listed (e.g., height, weight, IQ, blood pressure).

- **Categorical data** — represents a characteristic, like gender, marital status, country of birth, or favorite movie genre.
  - **Ordinal**: categories that have a natural order (e.g., student grades A, B, C; days of the week).
  - **Nominal**: categories with no inherent order (e.g., gender, marital status, country of birth).

Below, we classify a handful of everyday variables to make this concrete.


In [ ]:
# Demonstration: classify a handful of everyday variables into the four
# buckets above (Numerical-Discrete, Numerical-Continuous, Categorical-Ordinal,
# Categorical-Nominal). We store our classifications in a small pandas
# DataFrame just to make the mapping easy to read.
variable_types = pd.DataFrame({
    "variable": [
        "Person's height (cm)",
        "Number of siblings",
        "T-shirt size (S/M/L/XL)",
        "Eye color",
        "Blood pressure (mmHg)",
        "Number of cars owned",
    ],
    "classification": [
        "Numerical - Continuous (a measurement)",
        "Numerical - Discrete (a count)",
        "Categorical - Ordinal (S < M < L < XL)",
        "Categorical - Nominal (no natural order)",
        "Numerical - Continuous (a measurement)",
        "Numerical - Discrete (a count)",
    ]
})

variable_types


## 3. Data Sets, Samples, and Variables

- **Data set / Sample**: The collection of data taken for a study is called a **sample**.
  - Example: if you measured the **weights of five packages** and got `12, 22, 22, 68, 3` (in pounds), those five numbers *are* your data sample.
  - The same items could instead be recorded by their **corresponding sizes** — `medium, medium, medium, large, small` — which is just another representation of the same underlying data (you'd need a size definition, e.g., small < 15 lb, somewhere).

- **Variable**: A variable is any characteristic or numerical value that **varies from individual to individual**.
  - It can represent a **count** (e.g., number of siblings) or a **measurement** (e.g., weight).
  - It can be **categorical**, where an individual is placed into a group (e.g., package size).
  - The actual recorded values for a variable, across all individuals, *are* the data.

Let's build this exact example as a small table.


In [ ]:
# Recreate the "package weights" example from lecture as a small DataFrame.
# Two columns represent two different variables describing the SAME five
# packages: one numerical (weight in lbs) and one categorical (size).
packages = pd.DataFrame({
    "package_id": [1, 2, 3, 4, 5],
    "weight_lbs": [12, 22, 22, 68, 3],       # numerical (continuous) variable
    "size": ["medium", "medium", "medium", "large", "small"],  # categorical (ordinal) variable
})

packages


## 4. Population vs. Sample

- **Population**: your focused group of individuals/entities related to your query/research — e.g., a group of people, cities, animals, and so on.

Some sample research questions and the (very large) populations they imply:

- What do **Americans** think about the president's foreign policy?
- What percentage of planted crops in **Wisconsin** did deer destroy last year?
- What's the prognosis for **breast cancer patients** taking a new experimental drug?
- What percentage of **all cereal boxes** get filled according to specification?

> Many times researchers want to study and make conclusions about a broad population, but — to save time, money, or just because they don't know any better — they study only a narrowly defined *sample*. That shortcut can lead to big trouble when conclusions are drawn.

Because we usually can't collect data from an entire real-world population, we *simulate* one below: a population of 100,000 synthetic patients' systolic blood pressure readings. In practice, you'd never have this much data for free — this is just so we can see population vs. sample side by side.


In [ ]:
# Simulate a "population" of 100,000 patients' systolic blood pressure
# readings (mmHg), drawn from a normal distribution. In the real world we
# almost never have the entire population's data -- we're only doing this
# here so we can compare a population against a sample drawn from it.
population = np.random.normal(loc=120, scale=15, size=100_000)

# Draw a random SAMPLE of 50 patients from that population.
sample = np.random.choice(population, size=50, replace=False)

print(f"Population size: {len(population):,}")
print(f"Sample size:     {len(sample)}")

# Visualize: the full population's distribution, with our sample's values
# marked as ticks along the x-axis.
plt.hist(population, bins=60, color="#4C72B0", alpha=0.6, label="Population (n=100,000)")
plt.plot(sample, np.zeros_like(sample), '|', color="#DD8452", markersize=20, label="Sample (n=50)")
plt.xlabel("Systolic blood pressure (mmHg)")
plt.ylabel("Frequency")
plt.title("Population distribution with a sample drawn from it")
plt.legend()
plt.show()


## 5. Statistic vs. Parameter (and the Census)

- **Statistic**: a number that summarizes data collected as **a sample**. Examples: a *percentage* (60% of sampled households won more than 2 cars), an *average* (mean household income in a sample), a *median*, or a *percentile*.
- **Parameter**: if data is collected from the **entire population**, that collection is called a **census**. A single number summarizing a variable from census data is called a **parameter**.

In short: **statistics describe samples; parameters describe populations.** We compute statistics because parameters are usually unknowable in practice — but a good sample lets us *estimate* the parameter.


In [ ]:
# Compare the population mean (a PARAMETER, since we have the "entire
# population" here) against the sample mean (a STATISTIC, computed from
# just the 50 patients we sampled above).
population_mean = population.mean()   # parameter (mu)
sample_mean = sample.mean()           # statistic (x-bar)

print(f"Population mean (parameter, mu):  {population_mean:.2f} mmHg")
print(f"Sample mean (statistic, x-bar):   {sample_mean:.2f} mmHg")
print(f"Difference:                       {abs(population_mean - sample_mean):.2f} mmHg")


Notice the sample mean is *close* to the population mean but not identical — that gap is called **sampling error**, and it's expected. A statistic is an *estimate* of a parameter, not a guaranteed match. (Try re-running the simulation cell above with a larger sample size, like 5,000, and see the gap shrink.)


## 6. Bias

**Bias** is systematic favoritism that may be present in the data-collection process, resulting in lopsided, misleading results. Revisit the four research questions from Section 4 — each has room for bias to sneak in if the sample doesn't actually match the research query. Bias can occur in (at least) two common ways:

- **In the way the sample is selected.** Example: to estimate how much holiday shopping people in the U.S. plan to do, you stand in a mall on **Black Friday** with a clipboard. Your sample over-represents die-hard shoppers who braved the crowds that specific day — it's not representative of the whole population.
- **In the way the data is collected.** Example (question wording): *"Don't you think it would be a great investment in our future to support the local schools?"* versus *"Aren't you tired of paying money out of your pocket to educate other people's children?"* — both are asking about the same tax levy, but the wording pushes respondents toward opposite answers.

Let's demonstrate selection bias numerically using our simulated population.


In [ ]:
# Demonstrate SAMPLE-SELECTION BIAS: instead of sampling randomly from the
# whole population, we deliberately sample only from the top 10% of values
# (imagine this is like only surveying shoppers already inside the mall on
# Black Friday -- a convenient but non-representative subgroup).
top_decile_cutoff = np.percentile(population, 90)
biased_subgroup = population[population >= top_decile_cutoff]
biased_sample = np.random.choice(biased_subgroup, size=50, replace=False)

# For comparison, a genuinely random sample of the same size.
random_sample = np.random.choice(population, size=50, replace=False)

print(f"True population mean:      {population.mean():.2f} mmHg")
print(f"Random sample mean:        {random_sample.mean():.2f} mmHg   (close to the truth)")
print(f"Biased sample mean:        {biased_sample.mean():.2f} mmHg   (systematically too high!)")


The biased sample's mean is *far* from the true population mean — not because of bad luck, but because the sampling method **systematically** excluded most of the population. No amount of additional biased samples would fix this; only changing *how* the sample is drawn would.


## 7. Measures of Central Tendency

**Measures of central tendency** describe the central, or typical, value of a distribution — they help you find the "middle" or the "average" of a data set. We'll cover three: **mean**, **median**, and **mode**, using the package-weights example from Section 3: `12, 22, 22, 68, 3`.


### Mean

The **mean** (a.k.a. the average) is the most common statistic used to measure the center of a numerical data set.

- The mean is the sum of all values divided by the total number of observations.
- The mean of an entire **population** is called the **population mean** (symbol: $\mu$).
- The mean of a **sample** is called the **sample mean** (symbol: $\bar{x}$).

> ⚠️ The mean may not be a fair representation of the data, because the average is easily influenced by **outliers**.


In [ ]:
# Compute the mean of the package-weights example two ways: "by hand"
# (sum divided by count) and using numpy, to confirm they agree.
weights = np.array([12, 22, 22, 68, 3])

mean_by_hand = sum(weights) / len(weights)
mean_numpy = weights.mean()

print(f"Weights:            {weights}")
print(f"Mean (by hand):      {mean_by_hand}")
print(f"Mean (numpy):        {mean_numpy}")

plt.bar(range(1, len(weights) + 1), weights, color="#4C72B0")
plt.axhline(mean_numpy, color="#C44E52", linestyle="--", label=f"Mean = {mean_numpy}")
plt.xlabel("Package")
plt.ylabel("Weight (lbs)")
plt.title("Weights of five packages")
plt.legend()
plt.show()


### Median

The **median** is another way to measure the center of a numerical data set: it's the threshold where 50% of data points fall below it and 50% fall above it.


In [ ]:
# Compute the median "by hand" (sort the values, take the middle one) and
# with numpy, and confirm they match.
sorted_weights = np.sort(weights)
median_numpy = np.median(weights)

print(f"Sorted weights:  {sorted_weights}")
print(f"Median (numpy):  {median_numpy}")


### Mode

The **mode** represents the value with the highest concentration of points (i.e., the most frequently occurring value).


In [ ]:
# Compute the mode using scipy.stats -- the value that occurs most often.
mode_result = stats.mode(weights, keepdims=True)
print(f"Mode: {mode_result.mode[0]}  (appears {mode_result.count[0]} times)")


**Putting it together:** for `12, 22, 22, 68, 3`, the mean (25.4) is noticeably *higher* than the median and mode (both 22). That's because the outlier `68` pulls the mean upward, while the median and mode stay anchored to where most of the data actually sits. This is exactly why the mean "may not be a fair representation of the data" when outliers are present — always look at more than one measure of central tendency.


## 8. Variance and Standard Deviation

- **Variance**: the average of the squared differences from the mean. It measures *how spread out* the numbers are — on average, how far away values are from the average/mean. (Useful for comparing groups, e.g., a class with only A/B grades vs. a class with A/B/C grades — the second class has more spread.)
- **Standard deviation**: $\sqrt{\text{variance}}$. Another statistic measuring spread, in the *same units* as the original data (which is why it's usually easier to interpret than variance).

There's an important subtlety: the population and sample formulas differ slightly.

| | Population | Sample |
|---|---|---|
| # of subjects | $N$ | $n$ |
| Mean | $\mu = \dfrac{\sum_{i=1}^{N} x_i}{N}$ | $\bar{x} = \dfrac{\sum_{i=1}^{n} x_i}{n}$ |
| Variance | $\sigma^2 = \dfrac{\sum_{i=1}^{N}(x_i-\mu)^2}{N}$ | $S^2 = \dfrac{\sum_{i=1}^{n}(x_i-\bar{x})^2}{n-1}$ |
| Standard deviation | $\sigma = \sqrt{\dfrac{\sum_{i=1}^{N}(x_i-\mu)^2}{N}}$ | $S = \sqrt{\dfrac{\sum_{i=1}^{n}(x_i-\bar{x})^2}{n-1}}$ |

Note $S^2$ divides by $n-1$ instead of $n$ — this is the formula for an *unbiased* sample variance. (Taking $\sqrt{S^2}$ to get $S$ actually reintroduces a small bias, but $S$ is still the standard, widely-used estimator.)

In `numpy`, this distinction is controlled by the `ddof` (delta degrees of freedom) parameter: `ddof=0` (the default) computes the **population** formula; `ddof=1` computes the **sample** formula.


In [ ]:
# Compute BOTH the population-style and sample-style variance/std dev for
# our package weights, to see how the ddof parameter changes the result.
pop_variance = weights.var(ddof=0)   # divide by N
pop_std = weights.std(ddof=0)

sample_variance = weights.var(ddof=1)  # divide by (n - 1)
sample_std = weights.std(ddof=1)

summary = pd.DataFrame({
    "formula": ["Population (ddof=0)", "Sample (ddof=1)"],
    "variance": [pop_variance, sample_variance],
    "standard_deviation": [pop_std, sample_std],
})
summary


## 9. Percentile

The **percentile** reported for a given variable (score) is the percentage of values in the data set that fall **below** that threshold.

Example: if your score was reported to be at the **90th percentile**, that means **90%** of the other people who took the test scored lower than you did.


In [ ]:
# Recreate the 90th-percentile visualization from lecture using our
# simulated blood-pressure population: shade the bottom 90% of the
# distribution in one color and the top 10% in another.
from scipy.stats import norm

mu, sigma = population.mean(), population.std()
x = np.linspace(mu - 4 * sigma, mu + 4 * sigma, 500)
y = norm.pdf(x, mu, sigma)

p90 = np.percentile(population, 90)

plt.plot(x, y, color="black", linewidth=1)
plt.fill_between(x, y, where=(x <= p90), color="#4C72B0", alpha=0.6, label="90%")
plt.fill_between(x, y, where=(x > p90), color="#DD8452", alpha=0.6, label="10%")
plt.axvline(p90, color="black", linestyle="--", linewidth=1)
plt.text(p90, max(y) * 0.9, " 90th percentile", va="center")
plt.title("The 90th percentile of our simulated population")
plt.xlabel("Systolic blood pressure (mmHg)")
plt.yticks([])
plt.legend()
plt.show()

print(f"90th percentile value: {p90:.2f} mmHg")
print(f"25th / 50th / 75th percentiles: "
      f"{np.percentile(population, 25):.2f}, "
      f"{np.percentile(population, 50):.2f}, "
      f"{np.percentile(population, 75):.2f}")


## 10. Summary — Key Takeaways

- **Statistics** is the whole process of gathering evidence (data) to answer questions about the world — not just the numbers themselves.
- Data is either **numerical** (discrete or continuous) or **categorical** (ordinal or nominal).
- A **sample** is the data you actually collect; a **population** is the full group you ultimately care about.
- A **statistic** summarizes a sample; a **parameter** summarizes a population (a full population data collection is a **census**).
- **Bias** creeps in through *who* you sample and *how* you ask — both can badly mismatch your results to your real research question.
- **Mean**, **median**, and **mode** each describe the "center" of data differently, and can disagree sharply when outliers are present.
- **Variance** and **standard deviation** describe spread, with slightly different formulas for populations ($N$) vs. samples ($n-1$).
- A **percentile** tells you what fraction of a data set falls below a given value.

Now let's put these ideas to work.


---
## Exercises

Complete the five exercises below. Each one builds directly on a section above — if you get stuck, scroll back up to the matching section.

For code exercises, write your solution in the code cell provided (replace the `# TODO` comments). For short-answer questions, write your response in the markdown cell provided (double-click it to edit).


### Exercise 1 — Classify the Data Types

For each variable below, classify it as one of: **Numerical–Discrete**, **Numerical–Continuous**, **Categorical–Ordinal**, or **Categorical–Nominal**.

1. Height of a plant (cm)
2. T-shirt size (S, M, L, XL)
3. ZIP code
4. Number of pets owned
5. Blood type (A, B, AB, O)
6. Body temperature (°F)
7. Movie rating (1–5 stars)
8. Marital status (single, married, divorced)

*Hint: think carefully about #3 — just because something is written as digits doesn't automatically make it numerical in the statistical sense!*


In [ ]:
# TODO: Fill in your classification for each variable below.
# Replace the empty strings "" with one of:
#   "Numerical - Discrete", "Numerical - Continuous",
#   "Categorical - Ordinal", "Categorical - Nominal"

my_classifications = {
    "Height of a plant (cm)": "",
    "T-shirt size (S/M/L/XL)": "",
    "ZIP code": "",
    "Number of pets owned": "",
    "Blood type (A/B/AB/O)": "",
    "Body temperature (F)": "",
    "Movie rating (1-5 stars)": "",
    "Marital status": "",
}

for variable, classification in my_classifications.items():
    print(f"{variable:35s} -> {classification}")


### Exercise 2 — Central Tendency and Outliers

A clinic records the number of days it took each of 10 patients to recover from a minor procedure:

```
recovery_days = [4, 5, 5, 6, 7, 5, 4, 32, 6, 5]
```

1. Compute the **mean**, **median**, and **mode** of `recovery_days`.
2. In a markdown cell (or a comment), answer: which of the three measures best represents a "typical" patient's recovery time, and why? What is happening in this data set that makes the mean less trustworthy here?


In [ ]:
recovery_days = [4, 5, 5, 6, 7, 5, 4, 32, 6, 5]

# TODO: compute the mean, median, and mode of recovery_days.
# (You can use numpy / scipy.stats, or write the formulas by hand.)

mean_recovery = None    # TODO
median_recovery = None  # TODO
mode_recovery = None    # TODO

print("Mean:  ", mean_recovery)
print("Median:", median_recovery)
print("Mode:  ", mode_recovery)


*Your written answer for part 2 goes here.*


### Exercise 3 — Population vs. Sample

We've provided a synthetic **population** of 20,000 exam scores below (assume, for this exercise, that we somehow have data for *every* student who ever took this exam — i.e., this is a census).

1. Compute the population mean. Is this a **statistic** or a **parameter**?
2. Draw a random **sample** of 30 scores from the population (use the random seed given, for reproducibility) and compute its mean. Is this a statistic or a parameter?
3. How close are the two values? Re-run with a sample size of 300 — what happens to the gap, and why?


In [ ]:
np.random.seed(10)
exam_population = np.random.normal(loc=74, scale=9, size=20_000)
exam_population = np.clip(exam_population, 0, 100)  # scores must be between 0 and 100

# TODO: 1. Compute the population mean.
population_mean_score = None  # TODO

# TODO: 2. Draw a random sample of 30 scores (np.random.choice) and compute its mean.
sample_of_30 = None      # TODO
sample_mean_score = None  # TODO

print(f"Population mean: {population_mean_score}")
print(f"Sample mean (n=30): {sample_mean_score}")

# TODO: 3. Try again with a sample of size 300 and compare.


### Exercise 4 — Spot the Bias

For each scenario below, identify the type of bias present (**selection/sampling bias** or **question-wording bias**) and briefly explain, in one or two sentences, how you would fix the data-collection process.

**Scenario A:** A university wants to know how satisfied *all* students are with campus dining. Researchers stand in the campus food court and interview students between 12–1 PM on a Tuesday.

**Scenario B:** A city sends a poll asking: *"Don't you agree that the new fitness center, which will keep our students healthy, is a good use of tuition dollars?"*

**Scenario C:** A city wants to know whether *all* residents support a proposed bike lane. They mail paper surveys only to addresses within the downtown ZIP code, an area known for its cycling advocacy groups.


*Your answers here:*

- **Scenario A:** _(type of bias, and how to fix it)_
- **Scenario B:** _(type of bias, and how to fix it)_
- **Scenario C:** _(type of bias, and how to fix it)_


### Exercise 5 — Variance, Standard Deviation, and Percentiles

A professor records the following 12 exam scores:

```
scores = [55, 62, 75, 78, 80, 82, 85, 85, 88, 90, 95, 99]
```

1. Compute the **sample variance** and **sample standard deviation** (use $n-1$ in the denominator — i.e., `ddof=1`).
2. Compute the 25th, 50th, and 75th percentiles.
3. The **interquartile range (IQR)** is the 75th percentile minus the 25th percentile. Compute it, and in a sentence, explain what it tells you about this class's scores.


In [ ]:
scores = np.array([55, 62, 75, 78, 80, 82, 85, 85, 88, 90, 95, 99])

# TODO: 1. Compute the sample variance and sample standard deviation.
sample_var = None  # TODO
sample_std = None  # TODO

# TODO: 2. Compute the 25th, 50th, and 75th percentiles.
p25 = None  # TODO
p50 = None  # TODO
p75 = None  # TODO

# TODO: 3. Compute the IQR.
iqr = None  # TODO

print(f"Sample variance:     {sample_var}")
print(f"Sample std dev:      {sample_std}")
print(f"25th / 50th / 75th:  {p25}, {p50}, {p75}")
print(f"IQR:                 {iqr}")


*Your written interpretation of the IQR goes here.*


---
### Nice work!

You've now covered the core statistical vocabulary this course will build on: data types, samples vs. populations, statistics vs. parameters, bias, central tendency, spread, and percentiles. These ideas resurface constantly in knowledge discovery and data mining — from choosing how to summarize a feature, to spotting a biased training set, to deciding whether a difference between two groups is meaningful.

A companion notebook with **worked solutions** to these five exercises is available separately — try your best here first before checking it!
